# 3.9 · 数据泄漏 / Data Leakage

> **课程定位 / Where this fits**
> 第 9 课，**Part 3 · EDA 与数据预处理**。
> Lesson 9, **Part 3 · EDA & Preprocessing**.
>
> 前面几课反复强调"防泄漏"，这一课把它讲透。**数据泄漏(data leakage)** 是指：训练时用到了**预测时根本拿不到的信息**，导致离线指标好看、上线翻车。它是实战中**最隐蔽、代价最高**的错误，没有之一——很多"离线 98% 上线 70%"的事故都源于此。
> Earlier lessons kept saying "prevent leakage"; this one makes it explicit. **Data leakage** means training used **information unavailable at prediction time**, so offline metrics look great but production fails. It's the **most insidious and costly** mistake in practice — many "98% offline, 70% live" disasters trace back to it.
>
> 💼 **实战/面试视角**：泄漏是数据岗**必考**且能区分新手老手的题。"举一个数据泄漏的例子"几乎必问。
> 💼 **Practical/interview angle:** leakage is a must-ask that separates juniors from seniors. "Give an example of data leakage" is almost guaranteed.

> 💡 **面试相关 / Interview-relevant**
> - "什么是数据泄漏 / 举个例子"（出镜率 ★★★★★）
> - "目标泄漏：用了未来/结果信息"（★★★★★）
> - "预处理泄漏：在全数据上 fit"（★★★★★）
> - "时序数据为什么不能随机划分"（★★★★★）
> - "有重复实体(用户/病人)为什么要 GroupKFold"（★★★★）

---

## 学习目标 / Learning Objectives

1. 给出泄漏的**精确定义**：用了预测时拿不到的信息。
   Define leakage precisely: using info unavailable at prediction time.
2. 识别 **目标泄漏**（特征是结果/未来）。
   Spot **target leakage** (a feature that's a consequence/future).
3. 识别 **预处理泄漏**（在全数据上 fit），并用 Pipeline 根除。
   Spot **preprocessing leakage** (fitting on all data) and kill it with Pipelines.
4. 识别 **时序泄漏**（随机划分时序）与 **分组泄漏**（重复实体）。
   Spot **temporal leakage** (random-splitting time series) and **group leakage** (repeated entities).
5. 建立一套**防泄漏检查清单**。
   Build a leakage-prevention checklist.

## 目录 / TOC
1. [先建直觉：泄漏的定义](#1)
2. [💳 目标泄漏：用了结果 ⭐](#2)
3. [预处理/特征选择泄漏 ⭐](#3)
4. [时序泄漏：别随机划分 ⭐](#4)
5. [分组泄漏：重复实体 ⭐](#5)
6. [重复行泄漏 + 检查清单](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 先建直觉：泄漏的定义 / Intuition: Defining Leakage

一句话定义：**模型在训练时"偷看"了它在真实预测时不可能拥有的信息。**
One-line definition: **the model "peeked" during training at information it could not possibly have at real prediction time.**

判断泄漏的黄金问题：**"在我真正要做预测的那一刻，这条信息存在吗？"** 如果不存在（它是预测对象的结果、是未来才发生的、或来自测试集），用了它就是泄漏。
The golden test: **"At the exact moment I make a prediction, does this piece of information exist yet?"** If not (it's a consequence of the target, happens in the future, or comes from the test set), using it is leakage.

泄漏有四大类，本课逐一演示：
There are four main forms, demonstrated one by one:
1. **目标泄漏**：某特征其实是目标的"结果"或包含未来信息。
   **Target leakage:** a feature is a consequence of the target or contains future info.
2. **预处理泄漏**：缩放/填补/选特征时用了全部数据（含测试集）。
   **Preprocessing leakage:** scaling/imputing/selecting using all data (incl. test).
3. **时序泄漏**：时序数据随机划分，让模型"见过未来"。
   **Temporal leakage:** random-splitting time series lets the model "see the future".
4. **分组泄漏**：同一实体（用户/病人）的记录散落在 train 和 test 两边。
   **Group leakage:** records of the same entity split across train and test.


<a id="2"></a>
## 2. 目标泄漏：用了结果 ⭐ / Target Leakage

最典型的泄漏：某个特征看起来是"超强预测因子"，但它其实是目标**发生之后**才产生的。经典例子：预测"是否违约"时，用了"催收电话次数"——可催收是**违约的结果**，申请那一刻根本没有这个值。
The classic case: a feature looks like a "super predictor" but is actually generated **after** the target happens. Classic example: predicting "default" using "number of collection calls" — but collections are a **consequence** of default; at application time this value doesn't exist.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 信贷违约数据: 真实违约由 债务/收入 决定 / credit default depends on debt-to-income
n = 4000
income = rng.lognormal(10, 0.5, n); age = rng.uniform(22, 70, n); debt = rng.lognormal(8, 0.8, n)
logit = -3 + 2*(debt/income) + rng.normal(0, 0.5, n)
default = (1/(1+np.exp(-logit)) > 0.5).astype(int)
df = pd.DataFrame({"income":income,"age":age,"debt":debt,"default":default})

# 加一个泄漏特征 collection_calls: 它是违约的"结果"(违约者被反复催收) / a leaky feature
df["collection_calls"] = df["default"] * rng.poisson(8, n) + rng.poisson(0.2, n)
print(f"collection_calls 与 default 相关: {df['collection_calls'].corr(df['default']):.3f} (看似超强特征!)")
print("但它是违约'之后'才产生的 — 申请时根本没有这个值\n")

y = df["default"]
acc_clean = cross_val_score(RandomForestClassifier(random_state=0), df[["income","age","debt"]], y, cv=5).mean()
acc_leak  = cross_val_score(RandomForestClassifier(random_state=0), df[["income","age","debt","collection_calls"]], y, cv=5).mean()
print(f"干净特征 clean:   CV 准确率 = {acc_clean:.1%}")
print(f"+ 泄漏特征 leaky: CV 准确率 = {acc_leak:.1%}  ← 虚高 inflated!")
print("上线后 collection_calls 拿不到 → 模型实际只有干净特征水平 → 离线高、上线崩")


<a id="3"></a>
## 3. 预处理/特征选择泄漏 ⭐ / Preprocessing & Feature-Selection Leakage

第二类泄漏：**在划分前、用全部数据做了"会学参数"的预处理**——缩放(算均值)、填补(算中位数)、**特征选择**(用 y 挑特征)。这些都把测试集的信息渗进了训练。
The second form: doing **parameter-learning preprocessing on all data before splitting** — scaling (computes mean), imputation (computes median), **feature selection** (uses y to pick features). All of these seep test info into training.

下面用最极端的例子证明它有多危险：**5000 个纯噪声特征 + 完全随机的标签**。正确做法下准确率应该 ≈50%（瞎猜）。但如果**先在全数据上选特征再 CV**，5000 个噪声里总有 20 个"碰巧"和全部 y 相关——这份运气泄漏进 CV，给出虚高的分数。
An extreme demo of how dangerous it is: **5000 pure-noise features + completely random labels**. Done right, accuracy should be ≈50% (chance). But if you **select features on all data first, then CV**, among 5000 noise features ~20 will "happen" to correlate with the full y — that luck leaks into CV, inflating the score.


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# 纯噪声特征 + 随机标签: 真实准确率应≈50% / pure noise + random labels → should be chance
n2 = 200
X_noise = rng.normal(size=(n2, 5000))
y_rand = rng.integers(0, 2, n2)

# ❌ 错: 先在全数据(含将来的 test 折)上挑出 20 个"最相关"特征, 再 CV / WRONG
selector = SelectKBest(f_classif, k=20).fit(X_noise, y_rand)   # 这一步偷看了全部 y!
X_selected = selector.transform(X_noise)
acc_wrong = cross_val_score(LogisticRegression(max_iter=500), X_selected, y_rand, cv=5).mean()

# ✅ 对: 把特征选择放进 pipeline, CV 时每折只在该折的 train 上选 / RIGHT: selection inside pipeline
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=500))
acc_right = cross_val_score(pipe, X_noise, y_rand, cv=5).mean()

print("纯噪声特征 + 随机标签 (真实应 ≈ 50%):")
print(f"  ❌ 全数据先选特征: CV = {acc_wrong:.1%}  ← 虚高! 选特征时偷看了 test 折的 y")
print(f"  ✅ pipeline 内选:  CV = {acc_right:.1%}  ← 正确, 接近瞎猜")
print("教训: 任何'会看 y 或学参数'的步骤都必须放进 Pipeline, 让它只在每折 train 上 fit (3.12)")


<a id="4"></a>
## 4. 时序泄漏：别随机划分 ⭐ / Temporal Leakage

**时序数据绝不能随机划分**。随机划分会把"未来的点"放进训练集、"过去的点"放进测试集，等于让模型**见过未来**再去预测过去——上线时没有未来可见，性能崩塌。正确做法：**按时间切**，前段训练、后段测试（只用过去预测未来）。
**Never random-split time series.** A random split puts "future points" in train and "past points" in test, letting the model **see the future** before predicting the past — but live, there's no future to see, so it collapses. The fix: **split by time**, train on earlier, test on later (only the past predicts the future).


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 带上升趋势的时序数据 / time series with an upward trend
dates = pd.date_range("2025-01-01", periods=1000, freq="D")
trend = np.linspace(0, 10, 1000)
ts = pd.DataFrame({"feature": trend + rng.normal(0,1,1000), "target": trend + rng.normal(0,1,1000)})

# ❌ 随机划分: train 里混入了 test 时间点之后的"未来"样本 / random split leaks future
tr_idx, te_idx = train_test_split(np.arange(1000), test_size=0.3, random_state=0)
lr = LinearRegression().fit(ts.feature.values[tr_idx].reshape(-1,1), ts.target.values[tr_idx])
r2_random = r2_score(ts.target.values[te_idx], lr.predict(ts.feature.values[te_idx].reshape(-1,1)))

# ✅ 时间划分: 前 70% 训练, 后 30% 测试 / time-based split: past → future
split = 700
lr2 = LinearRegression().fit(ts.feature.values[:split].reshape(-1,1), ts.target.values[:split])
r2_time = r2_score(ts.target.values[split:], lr2.predict(ts.feature.values[split:].reshape(-1,1)))

print(f"随机划分 random: test R² = {r2_random:.3f}  (虚高 — train 里有 test 之后的未来点)")
print(f"时间划分 time:   test R² = {r2_time:.3f}  (真实 — 只用过去预测未来)")
print("时序数据永远按时间划分! 用 TimeSeriesSplit (3.10 详解)")


<a id="5"></a>
## 5. 分组泄漏：重复实体 ⭐ / Group Leakage

当数据里有**重复实体**（同一用户多次购买、同一病人多次就诊、同一设备多条日志）时，随机划分会把"同一实体的不同记录"分到 train 和 test 两边。这些记录高度相似，于是模型靠**"认出这个实体"**作弊，而不是学到真正的规律。真实场景里来的是**全新实体**，模型就废了。正确做法：**按实体分组划分**（`GroupKFold`，3.10 详解）。
When data has **repeated entities** (a user's multiple purchases, a patient's multiple visits, a device's many logs), a random split scatters one entity's records across train and test. These are highly similar, so the model cheats by **"recognizing the entity"** rather than learning the real pattern. Live, brand-new entities arrive and the model fails. The fix: **split by entity** (`GroupKFold`, detailed in 3.10).


In [ ]:
from sklearn.model_selection import GroupKFold

# 每个病人就诊 5 次, 同病人记录高度相似(共享病人固有特质) / 5 visits per patient
n_patients, visits_per = 200, 5
patient_id = np.repeat(np.arange(n_patients), visits_per)
patient_effect = rng.normal(0, 3, n_patients)                  # 每个病人的固有特质
X_med = (patient_effect[patient_id] + rng.normal(0, 0.5, n_patients*visits_per)).reshape(-1, 1)
y_med = (patient_effect[patient_id] + rng.normal(0, 0.5, n_patients*visits_per) > 0).astype(int)

# ❌ 随机 5 折: 同一病人的就诊散落在 train/test 两边 / random split mixes a patient's visits
acc_random = cross_val_score(RandomForestClassifier(random_state=0), X_med, y_med, cv=5).mean()
# ✅ GroupKFold: 保证同一病人整体只在一边 / keeps each patient entirely on one side
acc_group = cross_val_score(RandomForestClassifier(random_state=0), X_med, y_med,
                            cv=GroupKFold(5), groups=patient_id).mean()
print(f"随机划分 (同病人跨两边) random:  CV = {acc_random:.1%}  ← 虚高")
print(f"按病人分组划分 GroupKFold:       CV = {acc_group:.1%}  ← 真实")
print("随机划分时, test 的某次就诊在 train 里有'同一病人的另一次就诊'(极相似) → 模型靠认人作弊")
print("→ 凡有重复实体(用户/病人/设备)的数据, 必须 GroupKFold (3.10)")


<a id="6"></a>
## 6. 重复行泄漏 + 检查清单 / Duplicate Rows & Checklist

最后一种隐蔽泄漏：**数据里有完全重复的行**（采集/合并时产生）。不去重就划分，会让某些 test 行在 train 里有副本，模型"见过"它们，分数虚高。**划分前先 `drop_duplicates()`**。
A final subtle one: **exact duplicate rows** (from collection/merging). Splitting without deduping means some test rows have copies in train — the model has "seen" them, inflating the score. **`drop_duplicates()` before splitting.**


In [ ]:
base = pd.DataFrame({"x": rng.normal(size=100), "y": rng.integers(0,2,100)})
dup = pd.concat([base, base.sample(50, random_state=1)], ignore_index=True)   # 复制 50% 行

# 不去重直接划分: train 和 test 里会出现相同的 x 值 / split without dedup
X_tr, X_te, _, _ = train_test_split(dup[["x"]], dup["y"], test_size=0.3, random_state=0)
overlap = pd.merge(X_tr.assign(s="tr"), X_te.assign(s="te"), on="x", how="inner")
print(f"含重复数据直接划分: train 与 test 有 {len(overlap)} 个完全相同的 x → 模型见过, test 分数虚高")
print(f"防御: 先 df.drop_duplicates() (原 {len(dup)} 行 → 去重 {len(dup.drop_duplicates())} 行) 再划分")

print("""
\n=== 防泄漏检查清单 / Leakage-prevention checklist ===
1. 每个特征问: 预测那一刻它存在吗? 是不是目标的结果/未来? → 防目标泄漏
2. 所有'会学参数'的预处理(缩放/填补/选特征/编码)放进 Pipeline → 只 fit 训练折
3. 时序数据按时间划分(TimeSeriesSplit), 绝不随机
4. 有重复实体(用户/病人/设备)用 GroupKFold
5. 划分前 drop_duplicates()
6. 离线分数好得不真实? → 先怀疑泄漏""")


<a id="7"></a>
## 7. 小结 / Summary

```
泄漏定义: 训练用了'预测那一刻拿不到'的信息 → 离线虚高, 上线崩盘 (最隐蔽最贵的错)
黄金问题: 做预测的那一刻, 这条信息存在吗?
四类泄漏:
  目标泄漏: 特征是目标的结果/未来(如违约的'催收次数')
  预处理泄漏: 缩放/填补/选特征在全数据 fit → 用 Pipeline 只 fit 训练折
  时序泄漏: 随机划分时序 → 见过未来 → 按时间划分(TimeSeriesSplit)
  分组泄漏: 同实体记录跨 train/test → 靠认实体作弊 → GroupKFold
重复行: 划分前 drop_duplicates()
```

### 💡 面试速查 / Interview cheat-sheet
1. **泄漏 = 用了预测时拿不到的信息**；黄金问题"那一刻它存在吗"。
   Leakage = using info unavailable at prediction time; ask "does it exist at that moment?"
2. **目标泄漏**：特征是结果/未来（催收次数预测违约）。
   Target leakage: feature is a consequence/future (collections predicting default).
3. **预处理泄漏**：在全数据 fit；用 **Pipeline** 根除。
   Preprocessing leakage: fitting on all data; kill it with Pipelines.
4. **时序按时间划分、重复实体用 GroupKFold**。
   Time-split series; GroupKFold for repeated entities.
5. **离线好得离谱 → 先查泄漏**。
   Too-good offline → suspect leakage first.

### 下一节 / Next
**3.10 数据划分与交叉验证**——泄漏的解药正是"正确的划分"。K-fold、分层、分组、时序 CV，以及嵌套 CV 防调参泄漏。
**3.10 Splitting & Cross-Validation** — the antidote to leakage is correct splitting: K-fold, stratified, group, time-series CV, and nested CV.
